In [2]:
import torch
import torch.nn.functional as F

B = 1
T = 3
C = 4
n_head = 2
d_head = C // n_head

x = torch.arange(0, B * T * C, dtype=torch.float).view(B, T, C)
q = x.view(B, T, n_head, d_head).transpose(1, 2) # (B, n_head, T, d_head)
k = x.view(B, T, n_head, d_head).transpose(1, 2) # (B, n_head, T, d_head)
v = x.view(B, T, n_head, d_head).transpose(1, 2) # (B, n_head, T, d_head)

print(f"x:\n{x}")
print(f"q:\n{q}")
print(f"k:\n{k.transpose(-2, -1)}")

att_dot = q @ k.transpose(-2, -1) # (B, n_head, T, T)
print(f"att_dot:\n{att_dot}")
att_einsum = torch.einsum('bhid,bhjd->bhij', q, k) # (B, n_head, T, T)
print(f"att_einsum:\n{att_einsum}")

pattern_dot = F.softmax(att_dot, dim=-1) # (B, n_head, T, T)
print(f"pattern_dot:\n{pattern_dot}")

print(f"v:\n{v}")

y_dot = att_dot @ v # (B, n_head, T, d_head)
print(f"y_dot:\n{y_dot}")
y_einsum = torch.einsum('bhij,bhjd->bhid', att_einsum, v) # (B, n_head, T, d_head)
print(f"y_einsum:\n{y_einsum}")

y = y_einsum.transpose(1,2).contiguous().view(B, T, C)
print(f"y:\n{y}")

x:
tensor([[[ 0.,  1.,  2.,  3.],
         [ 4.,  5.,  6.,  7.],
         [ 8.,  9., 10., 11.]]])
q:
tensor([[[[ 0.,  1.],
          [ 4.,  5.],
          [ 8.,  9.]],

         [[ 2.,  3.],
          [ 6.,  7.],
          [10., 11.]]]])
k:
tensor([[[[ 0.,  4.,  8.],
          [ 1.,  5.,  9.]],

         [[ 2.,  6., 10.],
          [ 3.,  7., 11.]]]])
att_dot:
tensor([[[[  1.,   5.,   9.],
          [  5.,  41.,  77.],
          [  9.,  77., 145.]],

         [[ 13.,  33.,  53.],
          [ 33.,  85., 137.],
          [ 53., 137., 221.]]]])
att_einsum:
tensor([[[[  1.,   5.,   9.],
          [  5.,  41.,  77.],
          [  9.,  77., 145.]],

         [[ 13.,  33.,  53.],
          [ 33.,  85., 137.],
          [ 53., 137., 221.]]]])
pattern_dot:
tensor([[[[3.2932e-04, 1.7980e-02, 9.8169e-01],
          [5.3802e-32, 2.3195e-16, 1.0000e+00],
          [0.0000e+00, 2.9375e-30, 1.0000e+00]],

         [[4.2484e-18, 2.0612e-09, 1.0000e+00],
          [0.0000e+00, 2.6103e-23, 1.0000e+00],
